<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>

## RapidFire AI + Optuna: Adaptive SFT Hyperparameter Search

This tutorial shows how to use **RFOptuna** for Bayesian hyperparameter optimization
integrated into RapidFire's chunk-based training loop.

**Key difference from RFGridSearch / RFRandomSearch:**
- Grid/Random search decide all configs upfront
- **RFOptuna** uses Optuna's TPE sampler to suggest initial configs, then **prunes underperforming runs after each chunk** and replaces them with smarter suggestions

This notebook uses GPT-2 (124M params) so it runs on any machine.

### Enable Metric Loggers

In [1]:
import os
os.environ["RF_MLFLOW_ENABLED"] = "true"
os.environ["RF_TENSORBOARD_ENABLED"] = "true"
os.environ["RF_TRACKIO_ENABLED"] = "true"

### Imports

Note: `RFOptuna` requires the `optuna` package. Install with:
```bash
pip install optuna
```

In [2]:
from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    Range,
    RFOptuna,
    RFModelConfig,
    RFLoraConfig,
    RFSFTConfig,
)

### Load Dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

train_dataset = dataset["train"].select(range(64))
eval_dataset = dataset["train"].select(range(10, 11))
train_dataset = train_dataset.shuffle(seed=42)
eval_dataset = eval_dataset.shuffle(seed=42)

### Define Data Processing Function

In [4]:
def sample_formatting_function(example):
    """Format the dataset for GPT-2 while preserving original fields"""
    return {
        "text": f"Question: {example['instruction']}\nAnswer: {example['response']}",
        "instruction": example["instruction"],
        "response": example["response"],
    }

eval_dataset = eval_dataset.map(sample_formatting_function)
train_dataset = train_dataset.map(sample_formatting_function)

### Initialize Experiment

In [5]:
experiment = Experiment(experiment_name="exp-optuna-chatqa-tiny", mode="fit")

An experiment with the same name already exists. Created a new experiment 'exp-optuna-chatqa-tiny_2' with Experiment ID: 3 and Metric Experiment ID: 3 at /home/ubuntu/rapidfireai/rapidfire_experiments/exp-optuna-chatqa-tiny_2


### Define Config Template with Search Space

Instead of listing every combination (GridSearch) or sampling blindly (RandomSearch),
we define a **search space** using `Range(...)` and `List([...])` inside a single
`RFModelConfig`. Optuna's TPE sampler will intelligently explore this space.

**What Optuna controls:**
- `learning_rate`: Sampled from a continuous range
- `lora_alpha`: Sampled from a categorical list

**What stays fixed:**
- Model (`gpt2`), LoRA rank (`r=8`), batch size, etc.

In [6]:
config_template = RFModelConfig(
    model_name="gpt2",
    peft_config=RFLoraConfig(
        r=8,
        lora_alpha=List([16, 32, 64, 128]),
        lora_dropout=0.1,
        target_modules=["c_attn"],
        bias="none",
    ),
    training_args=RFSFTConfig(
        learning_rate=1e-3,
        lr_scheduler_type="linear",
        per_device_train_batch_size=2,
        max_steps=32,
        logging_steps=2,
        eval_strategy="steps",
        eval_steps=4,
        per_device_eval_batch_size=2,
        fp16=True,
        report_to="none",
    ),
    model_type="causal_lm",
    model_kwargs={
        "device_map": "auto",
        "torch_dtype": "float16",
        "use_cache": False,
    },
    formatting_func=sample_formatting_function,
)

### Create RFOptuna Config Group

Key parameters:
- **`n_initial=4`**: Start with 4 configs (sampled by TPE)
- **`budget=6`**: Allow up to 6 total configs (including replacements for pruned runs)
- **`objective="minimize:eval_loss"`**: Optuna minimizes eval loss to decide pruning
- **`sampler="tpe"`**: Tree-structured Parzen Estimator (Bayesian)
- **`pruner="median"`**: Prune runs performing worse than the median at each chunk

In [7]:
config_group = RFOptuna(
    configs=[config_template],
    trainer_type="SFT",
    n_initial=4,
    budget=6,
    objective="minimize:eval_loss",
    sampler="tpe",
    pruner="median",
    seed=42,
)

### Define Model Creation Function

In [8]:
def sample_create_model(model_config):
    """Function to create model object with GPT-2 adjustments"""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_type = model_config["model_type"]
    model_kwargs = model_config["model_kwargs"]

    if model_type == "causal_lm":
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if "gpt2" in model_name.lower():
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
        model.config.pad_token_id = model.config.eos_token_id

    return (model, tokenizer)

### Run Optuna-Powered Training

Behind the scenes:
1. `RFOptuna.get_runs()` asks Optuna's TPE sampler for 4 initial configs
2. RapidFire trains all 4 concurrently using chunk-based scheduling
3. After each chunk completes, the Optuna callback:
   - Reports the run's `eval_loss` to Optuna
   - Checks if Optuna's median pruner wants to prune the run
   - If pruned, suggests a replacement config (up to budget of 6)
4. RapidFire stops pruned runs and starts replacements automatically

In [9]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    eval_dataset,
    num_chunks=2,
    seed=42,
)

[I 2026-04-23 01:15:58,481] A new study created in memory with name: no-name-f4b2ca77-fbc6-4c87-9627-e3e1a8eb0ae6


Trackio: trackio show --project "exp-optuna-chatqa-tiny_2"
Started 2 worker processes successfully
Created workers


### Inspect Optuna Study Results

After training, the Optuna study object is accessible on the `RFOptuna` instance.
You can inspect which trials were completed, pruned, or failed.

In [10]:
from optuna.trial import TrialState

study = config_group._study

print(f"Number of trials: {len(study.trials)}")
done = [t for t in study.trials if t.state == TrialState.COMPLETE]
print(f"Completed trials: {len(done)}")
try:
    bt = study.best_trial
    print(f"Best trial value: {bt.value:.4f}")
    print(f"Best trial params: {bt.params}")
except (RuntimeError, ValueError) as exc:
    print(f"No best trial available yet ({exc}). Re-run fit or check logs.")
print()

for trial in study.trials:
    print(f"  Trial {trial.number}: state={trial.state.name}, "
          f"params={trial.params}, value={trial.value}")

Number of trials: 4
Completed trials: 4
Best trial value: 1.3708
Best trial params: {'peft_config.lora_alpha': 128}

  Trial 0: state=COMPLETE, params={'peft_config.lora_alpha': 32}, value=1.740251898765564
  Trial 1: state=COMPLETE, params={'peft_config.lora_alpha': 128}, value=1.3707669973373413
  Trial 2: state=COMPLETE, params={'peft_config.lora_alpha': 128}, value=1.38662588596344
  Trial 3: state=COMPLETE, params={'peft_config.lora_alpha': 16}, value=1.9466968774795532


### Get Results via RapidFire

In [11]:
results = experiment.get_results()
results

,run_id,step,loss,grad_norm,learning_rate,num_tokens,mean_token_accuracy,chunk number,num_epochs_completed,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_num_tokens,eval_mean_token_accuracy,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,1,2,2.8733,0.777430,0.000969,1164.0,0.455935,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,4,2.8138,0.647047,0.000906,2269.0,0.450904,0.0,0.0,2.431204,0.0239,41.771,41.771,2269.0,0.479866,NaN,NaN,NaN,NaN,NaN
2,1,6,2.5988,0.834244,0.000844,3370.0,0.474038,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,8,2.5918,0.926912,0.000781,4486.0,0.469496,0.0,0.0,2.232049,0.0201,49.818,49.818,4486.0,0.493289,NaN,NaN,NaN,NaN,NaN
4,1,10,2.3156,0.770061,0.000719,5587.0,0.497586,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,4,24,2.1579,0.639332,0.000281,4252.0,0.514400,1.0,0.0,1.995434,0.0202,49.624,49.624,4252.0,0.523490,NaN,NaN,NaN,NaN,NaN
60,4,26,2.3566,0.831825,0.000219,5019.0,0.494009,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,4,28,2.3460,0.610866,0.000156,5995.0,0.503253,1.0,0.0,1.960194,0.0203,49.338,49.338,5995.0,0.530201,NaN,NaN,NaN,NaN,NaN
62,4,30,2.1135,0.573814,0.000094,7188.0,0.522491,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### End Experiment

In [12]:
experiment.end()

Experiment exp-optuna-chatqa-tiny_2 ended
Workers stopped


---

### Comparison: RFOptuna vs RFGridSearch vs RFRandomSearch

| Feature | RFGridSearch | RFRandomSearch | RFOptuna |
|---|---|---|---|
| Config selection | All combos upfront | Random sample upfront | Bayesian (TPE) — learns from results |
| Pruning | Manual via IC Ops | Manual via IC Ops | Automatic (Median / Hyperband pruner) |
| Replacement | Manual clone-modify | Manual clone-modify | Automatic — new suggestions within budget |
| Search space | `List([...])` | `List([...])`, `Range(...)` | `List([...])`, `Range(...)` |
| Best for | Small discrete spaces | Large spaces, no adaptation | Large spaces, adaptive exploration |

<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Thanks for trying RapidFire AI! ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>